## Library

In [2]:
# Library
import time
import cv2
from ultralytics import YOLO


In [4]:
# YOLO
model = YOLO("yolo11n.pt")


In [3]:
# Start webcam
cap = cv2.VideoCapture(0)

while True:
    time.sleep(10)  # Capture for every 10 seconds

    retval, frame = cap.read()
    if not retval:
        print("Failed to grab frame.")
        break

    results = model(frame)
    for result in results:
        boxes = result.boxes
        for box in boxes: # Get bounding box
            x1, y1, x2, y2 = map(int, box.xyxy[0])  # Convert tensor to int (N, 4)
            conf = float(box.conf[0]) # Confidence score (N, 1)
            cls = int(box.cls[0]) # Class (Person: 0) (N, 1)

            if cls == 0 and conf > 0.5: # Crop the person
                cropped = frame[y1:y2, x1:x2]
                cv2.imshow("Cropped Person", cropped)
                cv2.imwrite("cropped_person.jpg", cropped)

    #annotated_frame = results[0].plot()
    #cv2.imshow("YOLO Annotated Frame", annotated_frame)

    # Quit with key ' '
    if cv2.waitKey(1) & 0xFF == ord(' '):
        break
    
cap.release()
cv2.destroyAllWindows()


0: 384x640 1 person, 77.1ms
Speed: 5.0ms preprocess, 77.1ms inference, 7.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 1 book, 62.4ms
Speed: 4.9ms preprocess, 62.4ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 3 books, 56.1ms
Speed: 3.6ms preprocess, 56.1ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 2 books, 486.1ms
Speed: 23.8ms preprocess, 486.1ms inference, 4.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 1 book, 359.5ms
Speed: 15.9ms preprocess, 359.5ms inference, 3.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 1 book, 65.4ms
Speed: 3.5ms preprocess, 65.4ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 1 book, 58.4ms
Speed: 4.5ms preprocess, 58.4ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 62.2ms
Speed: 3.0ms preprocess, 62.2ms i

: 

In [ ]:
import cv2
import torch
import torchvision.transforms as transforms
from torchvision.models import resnet18
from PIL import Image
import numpy as np

# Load pretrained ResNet
model = resnet18(pretrained=True)
model.eval()

# ImageNet class names
from torchvision.models import ResNet18_Weights
labels = ResNet18_Weights.DEFAULT.meta["categories"]

# Preprocessing
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Open webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Convert BGR to RGB
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img)
    input_tensor = preprocess(img_pil).unsqueeze(0)

    with torch.inference_mode():
        outputs = model(input_tensor)
        _, predicted = outputs.max(1)
        label = labels[predicted.item()]

    # Draw prediction on frame
    cv2.putText(frame, label, (20, 40), cv2.FONT_HERSHEY_SIMPLEX,
                1, (0, 255, 0), 2, cv2.LINE_AA)

    cv2.imshow("Real-Time ResNet", frame)
    if cv2.waitKey(1) & 0xFF == ord(' '):
        break

cap.release()
cv2.destroyAllWindows()

/Users/somnio.kimm/.pyenv/versions/3.10.13/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/somnio.kimm/.pyenv/versions/3.10.13/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


: 